# Steerling-8B — Concept Inspection & Steering

Make sure you're using the **Steerling (Python 3.13 ARM)** kernel.

In [1]:
import torch
from steerling import SteerlingGenerator, GenerationConfig

DEVICE = "mps"
MODEL_PATH = "/Users/vb/.cache/huggingface/hub/models--guidelabs--steerling-8b/snapshots/337e00164c67b3e458de12430246bd9e633568f7"

print("Loading model from local cache...")
gen = SteerlingGenerator.from_pretrained(MODEL_PATH, device=DEVICE)
print(f"Ready. Interpretable: {gen.is_interpretable}")

Loading model from local cache...
Ready. Interpretable: True


## 1. Basic generation

In [2]:
prompt = "The key to understanding neural networks is"

text = gen.generate(prompt, GenerationConfig(max_new_tokens=100, seed=42))
print(text)

 that they are made up of layers of interconnected nodes. Each node represents a neuron, which can process inputs and generate outputs based on the information given to it.

When it comes to deep learning, the algorithms used in this network have multiple layers of neurons. These hidden layers are responsible for extracting meaningful features from the input data. The final layer of the network is the output layer, where predictions or classifications are generated.

Deep learning models are trained using large amounts of labeled data. This means that the


## 2. Inspect active concepts

The model decomposes its hidden state into 33,732 known concept dimensions.
At each token position the top-16 concept IDs are active. Here we aggregate across
positions (mean activation) to see which concepts dominate for a given input.

In [ ]:
def get_concepts(text: str, topk: int = 20) -> list[tuple[int, float]]:
    """Return top-k known concept IDs averaged across token positions."""
    token_ids = gen.tokenizer.encode(text, add_special_tokens=False)
    x = torch.tensor([token_ids], dtype=torch.long, device=gen.device)

    with torch.inference_mode():
        # minimal_output=True forces the streaming top-k path, which populates
        # known_topk_indices/logits. minimal_output=False triggers the dense logit
        # path (return_logits=True) which skips top-k and leaves those fields None.
        _, outputs = gen.model(x, use_teacher_forcing=False, minimal_output=True)

    # shape: (1, T, K) -> (T, K)
    indices = outputs.known_topk_indices[0]
    logits  = outputs.known_topk_logits[0]

    scores: dict[int, list[float]] = {}
    for t in range(indices.shape[0]):
        for k in range(indices.shape[1]):
            cid   = int(indices[t, k].item())
            score = float(logits[t, k].item())
            scores.setdefault(cid, []).append(score)

    averaged = {cid: sum(v) / len(v) for cid, v in scores.items()}
    return sorted(averaged.items(), key=lambda x: x[1], reverse=True)[:topk]

In [6]:
text = "The economy is growing rapidly"
concepts = get_concepts(text, topk=20)

print(f"Top concepts for: '{text}'\n")
for rank, (cid, score) in enumerate(concepts, 1):
    print(f"  #{rank:2d}  [{cid:6d}]  {score:.4f}")

TypeError: 'NoneType' object is not subscriptable

## 3. Compare concepts across two prompts

Which concepts are shared vs. unique between two different texts?

In [ ]:
def compare_concepts(text_a: str, text_b: str, topk: int = 15):
    ca = dict(get_concepts(text_a, topk * 2))
    cb = dict(get_concepts(text_b, topk * 2))

    shared = sorted(set(ca) & set(cb), key=lambda c: ca[c] + cb[c], reverse=True)[:topk]
    only_a = sorted(set(ca) - set(cb), key=lambda c: ca[c], reverse=True)[:topk]
    only_b = sorted(set(cb) - set(ca), key=lambda c: cb[c], reverse=True)[:topk]

    print(f"A: '{text_a}'")
    print(f"B: '{text_b}'\n")

    print("── Shared ──────────────────────")
    for c in shared:
        print(f"  [{c:6d}]  A={ca[c]:.3f}  B={cb[c]:.3f}")

    print("\n── Only in A ────────────────────")
    for c in only_a:
        print(f"  [{c:6d}]  {ca[c]:.3f}")

    print("\n── Only in B ────────────────────")
    for c in only_b:
        print(f"  [{c:6d}]  {cb[c]:.3f}")


compare_concepts("The economy is booming", "The economy is collapsing")

## 4. Concept steering

`steer_known` maps `{concept_id: strength}`. Positive values amplify, negative suppress.
Compare baseline vs. steered generation on the same prompt + seed.

In [ ]:
def steer(prompt, steer_known=None, steer_unknown=None, max_new_tokens=80, seed=42):
    cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        seed=seed,
        steer_known=steer_known,
        steer_unknown=steer_unknown,
    )
    return gen.generate(prompt, cfg)


prompt = "The economy is"
concepts = get_concepts(prompt, topk=5)
top_id    = concepts[0][0]
second_id = concepts[1][0]

print(f"Top concepts: {concepts[:5]}\n")
print(f"── Baseline ──────────────────────────────")
print(steer(prompt))

print(f"\n── Amplify [{top_id}] x5 ─────────────────")
print(steer(prompt, steer_known={top_id: 5.0}))

print(f"\n── Suppress [{second_id}] x-5 ────────────")
print(steer(prompt, steer_known={second_id: -5.0}))

## 5. Concept activation heatmap (per token)

Show which concepts fire at each token position.

In [ ]:
def concept_heatmap(text: str):
    token_ids = gen.tokenizer.encode(text, add_special_tokens=False)
    tokens = [gen.tokenizer.decode([tid]) for tid in token_ids]
    x = torch.tensor([token_ids], dtype=torch.long, device=gen.device)

    with torch.inference_mode():
        _, outputs = gen.model(x, use_teacher_forcing=False, minimal_output=True)

    indices = outputs.known_topk_indices[0].cpu()  # (T, K)
    logits  = outputs.known_topk_logits[0].cpu()   # (T, K)

    print(f"Token-level top-3 concept activations for: '{text}'\n")
    for t, tok in enumerate(tokens):
        top3 = [(int(indices[t, k]), float(logits[t, k])) for k in range(3)]
        concept_str = "  ".join(f"[{cid}]={s:.2f}" for cid, s in top3)
        print(f"  {tok!r:20s}  {concept_str}")


concept_heatmap("Artificial intelligence is transforming medicine")

## 6. Sandbox — try your own steering

Pick any concept IDs from the inspection cells above and steer at different strengths.

In [ ]:
MY_PROMPT = "Climate change is"

# Step 1: see what concepts are active
for cid, score in get_concepts(MY_PROMPT, topk=10):
    print(f"  [{cid:6d}]  {score:.4f}")

In [ ]:
# Step 2: steer — edit concept_id and strength
concept_id = 0      # replace with an ID from above
strength   = 5.0    # positive = amplify, negative = suppress

print("Baseline:")
print(steer(MY_PROMPT))

print(f"\nSteered [{concept_id}] x{strength}:")
print(steer(MY_PROMPT, steer_known={concept_id: strength}))